In [4]:
import os
import h5py
import torch
from torch.utils.data import Dataset
import numpy as np

In [ ]:
# =============================================================================
# HELPER FUNCTION: Compute Topographic Features from DEM
# =============================================================================
# This is a plain Python function — no PyTorch needed here.
# It takes numpy arrays as input and returns numpy arrays as output.


def compute_topographic_features(
    dem, slope_deg, res=10.0, azimuth=315, angle_altitude=45
):

    # --- Step 1: Pad the DEM by 1 pixel on all sides ---
    # np.gradient() needs neighbors to compute derivatives at each pixel.
    # Without padding, edge pixels have missing neighbors and produce unstable results.
    # 'edge' mode repeats the border values outward.
    dem_padded = np.pad(dem, pad_width=1, mode="edge")

    # --- Step 2: Compute First-Order Gradients (Slope in X and Y) ---
    # np.gradient returns the rate of elevation change per meter.
    #   dx = slope going east  (columns)
    #   dy = slope going north (rows)
    # Note: np.gradient(array, spacing) returns (row_gradient, col_gradient)
    dy, dx = np.gradient(dem_padded, res)

    # --- Step 3: Compute Second-Order Gradients (Curvature) ---
    # Differentiate the first derivatives again to get curvature.
    #   d2x = rate of change of eastward slope  → east-west curvature
    #   d2y = rate of change of northward slope → north-south curvature
    d2y, _ = np.gradient(dy, res)  # We only need d2y from this call
    _, d2x = np.gradient(dx, res)  # We only need d2x from this call

    # --- Step 4: Crop back to original size ---
    # The padding added 1 row/column on each side, so we remove those borders
    # to get back to the original 128×128 shape.
    dx = dx[1:-1, 1:-1]
    dy = dy[1:-1, 1:-1]
    d2x = d2x[1:-1, 1:-1]
    d2y = d2y[1:-1, 1:-1]

    # --- Feature 1: Northness ---
    # aspect = compass direction the slope faces, in radians
    # We use arctan2(-dy, dx) because:
    #   - in image coordinates, the y-axis increases downward (south), not upward (north)
    #   - negating dy flips this to geographic north
    aspect = np.arctan2(-dy, dx)
    northness = np.cos(aspect)  # +1 = faces north, -1 = faces south

    # --- Feature 2: Curvature (Laplacian) ---
    # Positive curvature → convex terrain (ridges, hilltops)
    # Negative curvature → concave terrain (valleys, gullies)
    curvature = d2x + d2y

    # --- Feature 3: Hillshade ---
    # Simulates the brightness of terrain under a directional light source (the sun).
    # Uses the slope values from the raw ALOS band (slope_deg) instead of re-deriving slope.
    slope_rad = np.radians(slope_deg)
    azimuth_rad = np.radians(azimuth)
    altitude_rad = np.radians(angle_altitude)

    hillshade = np.sin(altitude_rad) * np.sin(slope_rad) + np.cos(
        altitude_rad
    ) * np.cos(slope_rad) * np.cos(azimuth_rad - aspect)
    hillshade = np.clip(hillshade, 0, 1)  # Clamp to [0, 1] — no negative light

    return northness, curvature, hillshade


# =============================================================================
# PYTORCH DATASET CLASS: LandslideDataset
# =============================================================================
# In TensorFlow you often loaded data with tf.data.Dataset or keras utilities.
# In PyTorch, the standard way is to define a Dataset class with these 3 methods:
#
#   __init__  → runs once when you create the dataset object (setup/configuration)
#   __len__   → tells PyTorch how many samples exist (used by DataLoader)
#   __getitem__ → loads and returns ONE sample by index (called repeatedly during training)
#
# PyTorch's DataLoader will call __getitem__ automatically in batches.


class LandslideDataset(Dataset):

    def __init__(self, img_dir, mask_dir=None, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

        # Build a list of all .h5 filenames in the folder
        # Extract just the numbers from image filenames → [1, 2, 3, ...]
        self.file_ids = sorted(
            [
                int(f.split("_")[1].split(".")[0])  # "image_1.h5" → 1
                for f in os.listdir(img_dir)
                if f.endswith(".h5")
            ]
        )

    def __len__(self):
        """
        Returns the total number of samples in the dataset.
        PyTorch's DataLoader calls this internally to know when one epoch is complete.
        """
        return len(self.file_ids)

    def __getitem__(self, idx):
        """
        Loads and returns a single (image, mask) pair by index.
        """

        file_id = self.file_ids[idx]  # e.g. 1
        img_name = f"image_{file_id}.h5"  # → "image_1.h5"
        mask_name = f"mask_{file_id}.h5"  # → "mask_1.h5"

        # --- Load image ---
        with h5py.File(os.path.join(self.img_dir, img_name), "r") as f:
            raw_image = f["img"][:]

        # --- Load mask ---
        if self.mask_dir is not None:
            with h5py.File(os.path.join(self.mask_dir, mask_name), "r") as f:
                mask = f["mask"][:]
        else:
            mask = np.zeros((128, 128), dtype=np.int64)
        eps = 1e-6

        # --- Extract Required Raw Bands (0-indexed) ---
        # ALOS PALSAR band layout (0-indexed in the .h5 file):
        blue = raw_image[:, :, 1]  # Band 2  — Blue
        green = raw_image[:, :, 2]  # Band 3  — Green
        red = raw_image[:, :, 3]  # Band 4  — Red
        nir = raw_image[:, :, 7]  # Band 8  — Near Infrared
        swir1 = raw_image[:, :, 10]  # Band 11 — Shortwave Infrared 1
        slope = raw_image[:, :, 12]  # Band 13 — Slope (degrees)
        dem = raw_image[:, :, 13]  # Band 14 — Digital Elevation Model (meters)

        # --- Compute Topographic Features from DEM ---
        northness, curvature, hillshade = compute_topographic_features(dem, slope)
        # --- Compute Spectral Indices ---
        ndvi = (nir - red) / (nir + red + eps)

        bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + eps)

        ndwi = (green - nir) / (green + nir + eps)

        # axis=-1 means the new axis is added at the END → shape: (128, 128, 8)
        image_8ch = np.stack(
            [dem, slope, northness, curvature, hillshade, ndvi, bsi, ndwi], axis=-1
        )  # Final shape: (128, 128, 8)

        # --- Reorder Axes for PyTorch ---
        # PyTorch expects tensors in (Channels, Height, Width) order,
        # but numpy stacked them as (Height, Width, Channels).
        # .transpose(2, 0, 1) moves axis 2 → position 0, keeping H and W after it.
        # In TensorFlow, images stay as (H, W, C) — this is the main difference!
        image_8ch = image_8ch.transpose(2, 0, 1)  # Shape: (8, 128, 128)

        # --- Convert numpy arrays to PyTorch Tensors ---
        # .float() → 32-bit float, required for most neural network layers
        # .long()  → 64-bit integer, required for CrossEntropyLoss mask labels
        image = torch.from_numpy(image_8ch).float()
        mask = torch.from_numpy(mask).long()

        # --- Apply Optional Augmentation ---
        # If you passed a transform function when creating the dataset, apply it here.
        # Example transforms: random flips, rotations, normalization.
        if self.transform:
            image, mask = self.transform(image, mask)

        return image, mask

In [ ]:
from torch.utils.data import DataLoader, random_split

# --- Create the dataset (pointing to separate img and mask folders) ---
full_dataset = LandslideDataset(
    img_dir="/mnt/e/Major project/Datasets/landslide4sense/TrainData/img",
    mask_dir="/mnt/e/Major project/Datasets/landslide4sense/TrainData/mask",
)

print("Total samples:", len(full_dataset))  # should print 3799

# --- Split into train and val ---
train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)


print("Train samples:", len(train_dataset))
print("Val samples:  ", len(val_dataset))

# --- Wrap in DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

# --- Verify everything works ---
images, masks = next(iter(train_loader))
print("Image shape:", images.shape)  # → torch.Size([16, 8, 128, 128])
print("Mask shape: ", masks.shape)  # → torch.Size([16, 128, 128])
print("Image dtype:", images.dtype)  # → torch.float32
print("Mask dtype: ", masks.dtype)  # → torch.int64

Total samples: 3799
Train samples: 3229
Val samples:   570
Image shape: torch.Size([16, 8, 128, 128])
Mask shape:  torch.Size([16, 128, 128])
Image dtype: torch.float32
Mask dtype:  torch.int64


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# ----- Part-1: U-Net Architecture -----


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),  ## out_channels means filters in keras, and keras figures out the in_channel itself
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


# Encoder Block: ConvBlock + MaxPool


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ConvBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)
        return features, pooled
        # return both:
        # features -> will be passed accross via skip connection to decoder
        # pooled -> goes down to the next encoder block


# Decoder Block: Upsampe + Concatenate skip + convBlock
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        # convTransposed2d doubles the spatial size: (64, 64) → (128, 128)

        self.conv = ConvBlock(out_channels * 2, out_channels)

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)  # concatenate along channel dimension
        x = self.conv(x)
        return x


# --- Full U-Net Model ---
class UNet(nn.Module):
    def __init__(self, in_channels=8, num_classes=2):
        """
        in_chennels : number of input channels - 8 for our dataset
        num_classes : 2 for binary segmentation (landslide / no-landslide)
        """

        super().__init__()

        # Encoder ( Contracting Path)

        self.enc1 = EncoderBlock(in_channels, 64)  # 8 → 64
        self.enc2 = EncoderBlock(64, 128)  # 64 → 128
        self.enc3 = EncoderBlock(128, 256)  # 128 → 256
        self.enc4 = EncoderBlock(256, 512)  # 256 → 512

        # Bottleneck (deepest point - no pooling)

        self.bottleneck = ConvBlock(512, 1024)  # 512 → 1024

        # Decoder (Expanding Path)

        self.dec4 = DecoderBlock(1024, 512)  # 1024 → 512
        self.dec3 = DecoderBlock(512, 256)  # 512 → 256
        self.dec2 = DecoderBlock(256, 128)  # 256 → 128
        self.dec1 = DecoderBlock(128, 64)  # 128 → 64

        # --- Final Output Layer ---
        self.output_conv = nn.conv2d(64, num_classes, kernel_size=1)
        # kernel size=1 -> 1x1 convolution, just maps 64 channels -> num_classes

    def forward(self, x):
        # ---Encoder ---
        skip1, x = self.enc1(x)  # skip for dec1, x: goes to enc2
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)  # skip for dec4, x: goes to bottleneck

        # --- Bottleneck ---
        x = self.bottleneck(x)

        # --- Decoder ---
        x = self.dec4(x, skip4)  # input from bottleneck, skip from enc4
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)  # input from dec2, skip from enc1

        # --- Final Output ---
        return self.output_conv(x)  # shape: (Batch, 2, 128, 128)

In [ ]:
import torch
import torch.nn as nn

# =============================================================================
# PART 2: LOSS FUNCTION
# =============================================================================
# BACKGROUND CONTEXT:
# In landslide segmentation, we face extreme "Class Imbalance". A satellite patch 
# might have 16,000 background pixels and only 200 landslide pixels. 
# Standard CrossEntropy treats every pixel equally, which tempts the model to 
# lazily guess "No Landslide" everywhere to achieve 99% accuracy. 
# Dice Loss fixes this by ignoring background pixels and forcing the model to 
# focus entirely on maximizing the regional overlap (Intersection) with actual landslides.
# =============================================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        # self.smooth prevents a catastrophic "Division by Zero" error.
        # If an image has 0 landslide pixels and the model correctly predicts 0,
        # the formula becomes 0/0 (which crashes). Adding 1.0 to top and bottom stabilizes it.
        self.smooth = smooth
 
    def forward(self, predictions, targets):
        # ---------------------------------------------------------------------
        # INPUT SHAPES:
        # predictions: (Batch, 2, H, W) -> Raw network outputs (Logits like [-2.4, 3.1])
        # targets:     (Batch, H, W)    -> Ground truth mask containing integers (0 or 1)
        # ---------------------------------------------------------------------
 
        # STEP 1: CONVERT LOGITS TO PROBABILITIES AND DROP THE BACKGROUND CHANNEL
        # 1. torch.softmax(..., dim=1) takes the raw channel values for every pixel 
        #    and crushes them into percentages (0.0 to 1.0) that sum to exactly 1.0 (100%).
        # 2. [:, 1, :, :] uses standard Python tuple indexing. It says:
        #    - First ':'  -> Keep all images in the batch.
        #    - '1'        -> Grab only Channel index 1 (the Landslide channel).
        #    - Last ':,:' -> Keep all Height rows and Width columns.
        # 3. DIMENSION DROPPING: Because we indexed with a single integer ('1') instead
        #    of a range ('1:2'), PyTorch drops the channel dimension entirely.
        #
        # FINAL SHAPE: (Batch, H, W) 
        # WHAT IS INSIDE: The decimal number stored inside each cell *is* the probability (0.0-1.0).
        probs = torch.softmax(predictions, dim=1)[:, 1, :, :] 
        
        # Convert integer targets (0 and 1) to floats so we can do decimal multiplication with probs
        targets_f = targets.float()
 
        # STEP 2: CALCULATE THE INTERSECTION (OVERLAP)
        # Multiplying 'probs' (0.0 to 1.0) by 'targets_f' (0 or 1) acts as a mask.
        # If the target pixel is 0 (Background), the product becomes 0.
        # If the target pixel is 1 (Landslide), it preserves the model's probability score.
        # .sum(dim=(1, 2)) flattens and sums all pixel values across Height and Width.
        # OUTPUT SHAPE: (Batch,) -> One total intersection score per image in the batch.
        intersection = (probs * targets_f).sum(dim=(1, 2))
 
        # STEP 3: THE DICE COEFFICIENT FORMULA
        # Dice = (2 * |Intersection|) / (|Predictions| + |Targets|)
        # This measures the percentage of overlap between the predicted map and real map.
        # A perfect match yields a Dice Score of 1.0. Complete failure yields 0.0.
        # OUTPUT SHAPE: (Batch,)
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2)) + self.smooth
        )
        
        # STEP 4: RETURN THE LOSS
        # Deep learning optimizers are built to MINIMIZE a loss value toward 0.
        # Since we want to MAXIMIZE our Dice Score toward 1.0, we calculate Loss = 1 - Dice.
        # .mean() averages this loss score across all images in the batch.
        return 1 - dice.mean() 


class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.5, ce_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight   = ce_weight
        self.dice = DiceLoss()
        
        # PyTorch's CrossEntropyLoss automatically accepts raw Logits (Batch, 2, H, W)
        # and integer Targets (Batch, H, W). It applies Softmax internally under the hood!
        self.ce   = nn.CrossEntropyLoss()
 
    def forward(self, predictions, targets):
        # WHY WE MIX THEM (50/50 Blend):
        # 1. CrossEntropy (CE) provides clean, highly stable mathematical gradients early 
        #    in training when the model is randomly guessing, but it suffers from class imbalance.
        # 2. Dice Loss ignores background pixels and focuses perfectly on landslide boundaries,
        #    but its gradients can be erratic and unstable when the model is completely untrained.
        # Combining them balances pixel-level confidence (CE) with global edge overlap (Dice).
        return (self.ce_weight   * self.ce(predictions, targets) +
                self.dice_weight * self.dice(predictions, targets))